# 📘 Textbook CRISP-DM Capstone: Brazilian E-Commerce by Olist

This notebook is designed as both a **complete data science project** and a **guided learning experience**.

## Business scenario

You are a data scientist for an e-commerce marketplace. Leadership wants a single analytical program that can answer:

1. **Who are our meaningful customer segments?**
2. **Which orders look unusual and deserve investigation?**
3. **Can we predict poor customer experiences before they become costly?**
4. **Which product categories tend to appear together?**
5. **How can we retrieve customers with similar purchase baskets without comparing every customer to every other customer?**

To answer those questions, we will use:

- exploratory data analysis
- preprocessing and feature engineering
- unsupervised clustering
- anomaly detection
- supervised classification
- association rule mining
- MinHash locality-sensitive hashing (LSH)
- CRISP-DM evaluation and synthesis

The project ends with a clean **conclusion phase that joins all evidence into business actions**.


## Why Olist?

The Kaggle Brazilian E-Commerce Public Dataset by Olist contains roughly 100,000 anonymized marketplace orders from 2016–2018 and separates data into customers, orders, items, payments, reviews, products, sellers, and geography.

That multi-table structure makes it unusually suitable for a full CRISP-DM capstone because the same business system supports descriptive, predictive, unsupervised, pattern-mining, and retrieval tasks.

### Real-data loading

If you have the Kaggle CSV files, place them beside the notebook or adjust `DATA_DIR`.

The notebook also contains a synthetic Olist-shaped fallback purely so every concept can run without Kaggle credentials. **Fallback metrics must never be reported as Kaggle benchmark results.**


# CRISP-DM Overview

CRISP-DM contains six phases:

1. **Business Understanding**
2. **Data Understanding**
3. **Data Preparation**
4. **Modeling**
5. **Evaluation**
6. **Deployment**

A crucial nuance: these are **not a strict waterfall**. If evaluation reveals leakage, poor features, or the wrong business target, you return to earlier phases.



> **Quiz**
>
> 1. Why is CRISP-DM better represented as a cycle than a straight line?
> 2. Which phase should define the business cost of a false positive?
> 3. If evaluation reveals target leakage, which earlier phase must be revisited?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. Because findings in later phases often force revisions to earlier assumptions, data, or modeling choices.
> 2. Business Understanding, because error costs come from the real decision context.
> 3. Usually Data Preparation first, and sometimes Business Understanding if the target itself was poorly defined.
>
> </details>


In [ ]:
# Imports
import math, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


# Phase 1 — Business Understanding

## 1.1 Translate the business question into analytical tasks

| Business question | Data science formulation | Output |
|---|---|---|
| Who are our meaningful customers? | clustering | customer segments |
| Which orders look abnormal? | anomaly detection | risk flags |
| Which orders may receive poor reviews? | supervised classification | probability / class |
| What categories co-occur? | association mining | support/confidence/lift rules |
| Which customers have similar baskets? | approximate similarity search | LSH candidates |

## 1.2 Define success before modeling

A textbook project explicitly defines success criteria **before** seeing model results.

Examples:

- Clustering: useful profiles + acceptable silhouette, not silhouette alone.
- Anomaly detection: investigator capacity and precision of flagged cases matter more than a visually interesting scatterplot.
- Supervised prediction: use F1/ROC-AUC if the low-review class is imbalanced.
- Association rules: lift must be paired with enough support.
- LSH: candidate recall and speed matter; approximate search trades some exactness for efficiency.

## 1.3 Risks and constraints

- The dataset is observational: association does not prove causation.
- Customer behavior can drift over time.
- Reviews may be missing or delayed.
- Customer IDs and order IDs have different business grains.
- Multiple items per order create join fanout risks.



> **Quiz**
>
> 1. Why should silhouette score not be the only clustering success criterion?
> 2. Why is accuracy often weak for predicting rare low-review events?
> 3. What is the business tradeoff behind LSH?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. A mathematically compact cluster may still be useless if it has no stable or actionable business interpretation.
> 2. A majority-class predictor can look accurate while failing to detect the minority class.
> 3. LSH gives up guaranteed exact nearest neighbors in exchange for much faster candidate retrieval.
>
> </details>


# Phase 2 — Data Understanding

## 2.1 Understand the grain before joining

Olist is multi-table data. The most important modeling mistake to avoid is confusing table grain.

Typical grains:

- customers: one row per customer ID
- orders: one row per order
- items: multiple rows per order
- payments: potentially multiple rows per order
- reviews: usually one review record per order
- products: one row per product

If you join `orders → items → payments` naively, you may multiply rows and inflate revenue.

### Textbook rule

> **Before every join, state the grain before and after the join.**


In [ ]:
# Real-data loader with Olist-shaped fallback.
# Set DATA_DIR to the folder containing the Kaggle CSVs.

from pathlib import Path
DATA_DIR = Path(".")

required = {
    "orders":"olist_orders_dataset.csv",
    "items":"olist_order_items_dataset.csv",
    "customers":"olist_customers_dataset.csv",
    "reviews":"olist_order_reviews_dataset.csv",
    "products":"olist_products_dataset.csv",
}

use_real = all((DATA_DIR/f).exists() for f in required.values())

if use_real:
    real_orders = pd.read_csv(DATA_DIR/required["orders"], parse_dates=[
        "order_purchase_timestamp","order_delivered_customer_date"
    ])
    real_items = pd.read_csv(DATA_DIR/required["items"])
    real_customers = pd.read_csv(DATA_DIR/required["customers"])
    real_reviews = pd.read_csv(DATA_DIR/required["reviews"])
    real_products = pd.read_csv(DATA_DIR/required["products"])

    # Build one order-level modeling table without join fanout.
    item_agg = real_items.groupby("order_id").agg(
        order_value=("price","sum"),
        freight_value=("freight_value","sum"),
        item_count=("order_item_id","count")
    ).reset_index()

    review_agg = real_reviews.groupby("order_id").agg(
        review_score=("review_score","mean")
    ).reset_index()

    orders = (
        real_orders
        .merge(real_customers[["customer_id","customer_unique_id","customer_state"]],on="customer_id",how="left")
        .merge(item_agg,on="order_id",how="left")
        .merge(review_agg,on="order_id",how="left")
    )
    orders["purchase_date"] = pd.to_datetime(orders["order_purchase_timestamp"])
    orders["delivery_days"] = (
        pd.to_datetime(orders["order_delivered_customer_date"]) -
        pd.to_datetime(orders["order_purchase_timestamp"])
    ).dt.days
    orders["status"] = orders["order_status"]
    orders["low_review"] = (orders["review_score"]<=2).astype(int)
    items = real_items.merge(
        real_products[["product_id","product_category_name"]],
        on="product_id",how="left"
    )[["order_id","product_category_name","price","freight_value"]]

    DATA_MODE = "REAL KAGGLE OLIST"
else:
    # Compact Olist-shaped fallback, generated deterministically.
    rng=np.random.default_rng(42)
    n_customers=2200; n_orders=8500
    customers=[f"C{i:05d}" for i in range(n_customers)]
    states=["SP","RJ","MG","RS","PR","BA","SC","GO"]
    cats=["health_beauty","bed_bath_table","sports_leisure","computers_accessories",
          "watches_gifts","housewares","toys","fashion_bags","telephony","books"]
    dates=pd.date_range("2017-01-01","2018-08-31",freq="D")

    orders=pd.DataFrame({
        "order_id":[f"O{i:06d}" for i in range(n_orders)],
        "customer_unique_id":rng.choice(customers,n_orders),
        "purchase_date":pd.to_datetime(rng.choice(dates,n_orders)),
        "customer_state":rng.choice(states,n_orders),
        "status":rng.choice(["delivered","shipped","canceled","invoiced"],n_orders,p=[.92,.04,.025,.015]),
    })
    orders["delivery_days"]=np.maximum(1,rng.normal(11,5,n_orders).round()).astype(int)

    rows=[]
    for oid in orders["order_id"]:
        k=int(rng.choice([1,2,3,4],p=[.74,.18,.06,.02]))
        for c in rng.choice(cats,k,replace=False):
            rows.append([oid,c,max(8,rng.lognormal(4.4,.45)),max(2,rng.lognormal(2.7,.35))])
    items=pd.DataFrame(rows,columns=["order_id","product_category_name","price","freight_value"])

    agg=items.groupby("order_id").agg(
        order_value=("price","sum"),
        freight_value=("freight_value","sum"),
        item_count=("product_category_name","size")
    ).reset_index()
    orders=orders.merge(agg,on="order_id")
    penalty=np.clip((orders["delivery_days"]-10)/20,0,1.5)
    review=4.5-1.6*penalty-orders["status"].eq("canceled").astype(float)*1.5+rng.normal(0,.7,n_orders)
    orders["review_score"]=np.clip(np.rint(review),1,5)
    orders["low_review"]=(orders["review_score"]<=2).astype(int)
    DATA_MODE="SYNTHETIC OLIST-SHAPED FALLBACK"

orders["purchase_month"]=pd.to_datetime(orders["purchase_date"]).dt.to_period("M").astype(str)
print(DATA_MODE, orders.shape, items.shape)
display(orders.head())


## 2.2 Programmatic EDA

A good first pass checks:

- shape and grain
- dtypes
- missingness
- duplicate keys
- target balance
- time range
- distributions
- impossible values
- category cardinality


In [ ]:
eda = pd.DataFrame({
    "dtype": orders.dtypes.astype(str),
    "missing": orders.isna().sum(),
    "missing_pct": orders.isna().mean(),
    "nunique": orders.nunique()
})
display(eda)

print("duplicate order_id:", orders["order_id"].duplicated().sum())
print("date range:", orders["purchase_date"].min(), "→", orders["purchase_date"].max())
print("low-review rate:", orders["low_review"].mean())
display(orders[["order_value","freight_value","delivery_days","review_score"]].describe())



> **Quiz**
>
> 1. Why can joining two one-to-many tables through orders inflate revenue?
> 2. What is the difference between missingness and invalid values?
> 3. Why should time range be inspected before train/test splitting?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. Because an order can appear multiple times in both tables, creating a Cartesian multiplication at the order level.
> 2. Missingness means no value is present; invalid values are present but violate domain/business rules.
> 3. Because temporal coverage and drift may make a random split unrealistic or leak future patterns into training.
>
> </details>


# Phase 3 — Data Preparation

## 3.1 Create analysis-specific tables

A nuanced CRISP-DM project rarely has one universal modeling table.

We need different representations:

- **RFM customer table** for clustering
- **order-level table** for anomaly detection and supervised ML
- **order basket representation** for association rules
- **customer set representation** for MinHash LSH

This is an important lesson: preprocessing depends on the analytical task.


In [ ]:
# Basic quality rules
orders = orders.drop_duplicates("order_id").copy()
orders = orders[orders["order_value"].fillna(0) >= 0]
orders = orders[orders["freight_value"].fillna(0) >= 0]

# Modeling-safe engineered features
orders["freight_ratio"] = orders["freight_value"] / (orders["order_value"] + orders["freight_value"] + 1e-9)
orders["is_delivered"] = orders["status"].eq("delivered").astype(int)
orders["purchase_month_num"] = pd.to_datetime(orders["purchase_date"]).dt.month
orders["purchase_weekday"] = pd.to_datetime(orders["purchase_date"]).dt.weekday

display(orders.head())


## 3.2 Leakage

Suppose the task is to predict a low review **before the customer submits the review**.

Then you must not use:

- `review_score`
- review text
- fields created after the review arrives

Whether `delivery_days` is valid depends on the prediction moment:

- **Before shipping:** invalid; future leakage.
- **After delivery but before review:** potentially valid.
- **At order placement:** invalid.

This is why feature validity is a **business-time question**, not just a correlation question.



> **Quiz**
>
> 1. Why is preprocessing task-specific rather than universal?
> 2. If low-review prediction occurs at checkout, may actual delivery_days be used?
> 3. Why should scaling for K-Means be fit only on the data being clustered?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. Different algorithms need different grains, targets, representations, and valid information.
> 2. No. Actual delivery duration occurs in the future and would leak information unavailable at checkout.
> 3. Because scale determines Euclidean distances; external/future information should not influence that geometry.
>
> </details>


# Phase 4 — Modeling

CRISP-DM calls this one phase, but our capstone contains five very different modeling paradigms.

The goal is to understand **what each method answers** rather than treating all algorithms as interchangeable.


## 4A — Unsupervised Learning: Customer Clustering

### RFM representation

- **Recency:** how long since the customer's latest purchase
- **Frequency:** number of orders
- **Monetary:** total spend

K-Means minimizes within-cluster squared Euclidean distance. Therefore:

- features must be on comparable scales;
- skewed monetary data often benefits from a log transform;
- `k` should be selected using both quantitative and business criteria.


In [ ]:
snapshot = orders["purchase_date"].max() + pd.Timedelta(days=1)

rfm = orders.groupby("customer_unique_id").agg(
    recency=("purchase_date", lambda x:(snapshot-pd.to_datetime(x).max()).days),
    frequency=("order_id","nunique"),
    monetary=("order_value","sum")
).reset_index()

X_rfm = np.log1p(rfm[["recency","frequency","monetary"]])
scaled = StandardScaler().fit_transform(X_rfm)

scores={}
for k in range(2,7):
    labels=KMeans(n_clusters=k,random_state=RANDOM_STATE,n_init=20).fit_predict(scaled)
    scores[k]=silhouette_score(scaled,labels)

best_k=max(scores,key=scores.get)
km=KMeans(n_clusters=best_k,random_state=RANDOM_STATE,n_init=20)
rfm["cluster"]=km.fit_predict(scaled)

print("silhouette by k:",scores)
print("selected k:",best_k)
display(rfm.groupby("cluster")[["recency","frequency","monetary"]].mean().round(1))



> **Quiz**
>
> 1. Why can raw monetary value dominate K-Means?
> 2. What does a higher silhouette score generally mean?
> 3. Why should a business analyst inspect cluster profiles after choosing k?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. Because K-Means uses Euclidean distance, so a high-variance feature can dominate the geometry.
> 2. Points are, on average, better matched to their own cluster than neighboring clusters.
> 3. To verify clusters are stable, understandable, and actionable rather than merely mathematically separated.
>
> </details>


## 4B — Anomaly / Outlier Detection

### Isolation Forest

Isolation Forest recursively partitions the feature space. Rare or unusual observations tend to be isolated with fewer splits.

We use:

- order value
- freight value
- delivery days
- item count

The contamination parameter controls roughly how many cases are flagged; it should reflect review capacity or expected anomaly prevalence, not be selected just to make an attractive chart.


In [ ]:
anomaly_features = orders[["order_value","freight_value","delivery_days","item_count"]].copy()
anomaly_features = anomaly_features.fillna(anomaly_features.median())

X_anom = StandardScaler().fit_transform(np.log1p(anomaly_features.clip(lower=0)))

iso = IsolationForest(
    n_estimators=250,
    contamination=.015,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
orders["anomaly_flag"] = (iso.fit_predict(X_anom)==-1).astype(int)
orders["anomaly_score"] = -iso.score_samples(X_anom)

print("flagged rate:",orders["anomaly_flag"].mean())
display(
    orders.nlargest(10,"anomaly_score")[
        ["order_id","order_value","freight_value","delivery_days","item_count","anomaly_score"]
    ]
)



> **Quiz**
>
> 1. Does an anomaly flag prove fraud or bad data?
> 2. What business input should influence contamination?
> 3. Why might log1p help order-value anomaly detection?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. No. It only indicates unusualness relative to the learned feature distribution.
> 2. Expected prevalence and/or analyst investigation capacity.
> 3. It compresses heavy right tails so extreme but legitimate high-value orders do not dominate scaling as strongly.
>
> </details>


## 4C — Supervised Machine Learning: Predict Low Review Risk

Target:

`low_review = review_score <= 2`

We compare:

- Logistic Regression
- Random Forest

### Leakage-safe pipeline

All imputation, scaling, and one-hot encoding are inside a scikit-learn Pipeline.

This matters because preprocessing learned from the full dataset would leak test-set distributional information.


In [ ]:
features = [
    "order_value","freight_value","delivery_days","item_count",
    "freight_ratio","customer_state","status"
]
model_df = orders.dropna(subset=["low_review"]).copy()

X = model_df[features]
y = model_df["low_review"].astype(int)

num = ["order_value","freight_value","delivery_days","item_count","freight_ratio"]
cat = ["customer_state","status"]

pre = ColumnTransformer([
    ("num", Pipeline([
        ("impute",SimpleImputer(strategy="median")),
        ("scale",StandardScaler())
    ]), num),
    ("cat", Pipeline([
        ("impute",SimpleImputer(strategy="most_frequent")),
        ("onehot",OneHotEncoder(handle_unknown="ignore"))
    ]), cat)
])

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=.25,stratify=y,random_state=RANDOM_STATE
)

results=[]
trained={}
for name,model in {
    "Logistic Regression":LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RANDOM_STATE),
    "Random Forest":RandomForestClassifier(
        n_estimators=300,max_depth=10,class_weight="balanced",
        random_state=RANDOM_STATE,n_jobs=-1
    )
}.items():
    pipe=Pipeline([("prep",pre),("model",model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    proba=pipe.predict_proba(X_test)[:,1]
    results.append({
        "Model":name,
        "F1":f1_score(y_test,pred),
        "ROC_AUC":roc_auc_score(y_test,proba)
    })
    trained[name]=pipe

results_df=pd.DataFrame(results).sort_values("F1",ascending=False)
display(results_df)



> **Quiz**
>
> 1. Why is class_weight='balanced' useful here?
> 2. Why use both F1 and ROC-AUC?
> 3. Why is a Pipeline safer than preprocessing before train/test split?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. It increases the penalty for mistakes on the minority class without manually duplicating rows.
> 2. F1 summarizes precision/recall at a threshold; ROC-AUC measures ranking quality across thresholds.
> 3. Pipeline steps are fit only on training data during fitting/CV, reducing preprocessing leakage.
>
> </details>


## 4D — Associative Rule Mining

We convert each order into a set of product categories.

For a rule `A → B`:

- **Support:** fraction of baskets containing both A and B
- **Confidence:** P(B | A)
- **Lift:** confidence divided by the baseline probability of B

Interpretation:

- lift > 1: positive association
- lift ≈ 1: roughly independent
- lift < 1: negative association

### Nuance

A huge lift on a tiny number of transactions is often less useful than a moderate lift with meaningful support.


In [ ]:
# Binary basket rule mining for category pairs
basket_series = (
    items.dropna(subset=["product_category_name"])
    .groupby("order_id")["product_category_name"]
    .apply(lambda x:set(x))
)
baskets=list(basket_series)

item_count={}
pair_count={}
N=len(baskets)

for basket in baskets:
    for a in basket:
        item_count[a]=item_count.get(a,0)+1
    for a in basket:
        for b in basket:
            if a!=b:
                pair_count[(a,b)]=pair_count.get((a,b),0)+1

rules=[]
for (a,b),cnt in pair_count.items():
    support=cnt/N
    confidence=cnt/item_count[a]
    lift=confidence/(item_count[b]/N)
    if support>=.005 and confidence>=.05:
        rules.append([a,b,support,confidence,lift])

rules_df=pd.DataFrame(
    rules,
    columns=["antecedent","consequent","support","confidence","lift"]
).sort_values(["lift","support"],ascending=False)

display(rules_df.head(15))



> **Quiz**
>
> 1. Why can confidence be misleading without lift?
> 2. What does lift = 2 mean intuitively?
> 3. Why filter on minimum support?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. A consequent that is already very common can produce high confidence even without a meaningful association.
> 2. B occurs after A at about twice the rate expected from B's baseline frequency.
> 3. To avoid prioritizing unstable rules based on extremely rare baskets.
>
> </details>


## 4E — Sub-Linear Search: MinHash Locality-Sensitive Hashing

### The exact problem

Suppose every customer is represented by the **set of categories they have purchased**.

Exact similarity uses Jaccard:

`J(A,B) = |A ∩ B| / |A ∪ B|`

Comparing one customer against all customers is O(N). Comparing all pairs is O(N²).

### MinHash

MinHash creates a compact signature whose collision probability estimates Jaccard similarity.

### LSH

The signature is split into bands. Similar signatures are likely to collide in at least one band, producing a small candidate set.

This is why LSH is called **sub-linear approximate search**: we avoid checking every possible record.


In [ ]:
customer_categories = (
    orders[["order_id","customer_unique_id"]]
    .merge(items[["order_id","product_category_name"]],on="order_id")
    .dropna(subset=["product_category_name"])
    .groupby("customer_unique_id")["product_category_name"]
    .apply(set)
)

customer_categories = customer_categories[customer_categories.map(len)>=2]
customer_ids=list(customer_categories.index)[:1000]

rng=np.random.default_rng(42)
PRIME=4294967311
NUM_PERM=64
BANDS=16
ROWS_PER_BAND=NUM_PERM//BANDS

a_coeff=rng.integers(1,PRIME-1,NUM_PERM,dtype=np.int64)
b_coeff=rng.integers(0,PRIME-1,NUM_PERM,dtype=np.int64)

def stable_hash(text):
    h=2166136261
    for byte in text.encode("utf-8"):
        h=(h^byte)*16777619
        h&=0xffffffff
    return h

def minhash_signature(values):
    x=np.array([stable_hash(v) for v in values],dtype=np.int64)
    return tuple(int(np.min((a*x+b)%PRIME)) for a,b in zip(a_coeff,b_coeff))

signatures={cid:minhash_signature(customer_categories[cid]) for cid in customer_ids}

buckets={}
for cid,sig in signatures.items():
    for band in range(BANDS):
        start=band*ROWS_PER_BAND
        key=(band,sig[start:start+ROWS_PER_BAND])
        buckets.setdefault(key,set()).add(cid)

def candidates(cid):
    sig=signatures[cid]
    out=set()
    for band in range(BANDS):
        start=band*ROWS_PER_BAND
        out |= buckets.get((band,sig[start:start+ROWS_PER_BAND]),set())
    out.discard(cid)
    return out

def jaccard(a,b):
    return len(a&b)/len(a|b) if a|b else 0.0

query=None
cand=set()
for cid in customer_ids:
    c=candidates(cid)
    if c:
        query=cid; cand=c; break

print("query customer:",query)
print("total indexed customers:",len(customer_ids))
print("LSH candidate count:",len(cand))

if query is not None:
    neighbors=sorted(
        [(c,jaccard(customer_categories[query],customer_categories[c])) for c in cand],
        key=lambda x:x[1],reverse=True
    )[:10]
    display(pd.DataFrame(neighbors,columns=["customer","exact_jaccard_after_LSH"]))



> **Quiz**
>
> 1. Why is MinHash especially appropriate for sets?
> 2. What happens if LSH uses more bands with fewer rows per band?
> 3. Why calculate exact Jaccard after retrieving LSH candidates?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. MinHash collision probability is directly related to Jaccard similarity between sets.
> 2. The system usually becomes more permissive, increasing candidate recall but also false-positive candidates.
> 3. LSH is a candidate generator; exact scoring reranks the small candidate set accurately.
>
> </details>


# Phase 5 — Evaluation

CRISP-DM evaluation asks a broader question than “which metric is highest?”

We evaluate three levels:

## Technical evaluation

- clustering: silhouette + stability + interpretable profiles
- anomaly detection: inspection workload + anomaly validity
- supervised ML: F1, ROC-AUC, confusion matrix
- association rules: support, confidence, lift
- LSH: candidate count, exact Jaccard of retrieved neighbors, speed/recall tradeoff

## Business evaluation

- Are customer segments actionable?
- Can anomaly investigations reduce losses or operational mistakes?
- Is low-review risk available early enough to intervene?
- Do cross-sell rules support merchandising?
- Is similar-customer retrieval useful for recommendation or case retrieval?

## Process evaluation

- Was there leakage?
- Were joins performed at the correct grain?
- Are assumptions documented?
- Would another analyst reproduce the result?


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_pipe = trained[best_name]
pred = best_pipe.predict(X_test)
cm = confusion_matrix(y_test,pred)

print("Best supervised model:",best_name)
print("Confusion matrix:")
display(pd.DataFrame(cm,index=["Actual 0","Actual 1"],columns=["Pred 0","Pred 1"]))

print("\nSelected clustering k:",best_k)
print("Best silhouette:",round(scores[best_k],3))
print("Anomaly rate:",round(orders["anomaly_flag"].mean(),4))
print("Association rules retained:",len(rules_df))



> **Quiz**
>
> 1. Why is a technically strong model not automatically deployable?
> 2. What is one evaluation question common to every method in this notebook?
> 3. Why should anomaly samples be manually inspected?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. It may be too late for the business decision, too costly, unfair, unstable, or operationally unusable.
> 2. Does this output improve the intended business decision without relying on leakage or invalid assumptions?
> 3. Because unsupervised anomaly scores indicate unusualness, not the business meaning or cause of the unusual case.
>
> </details>


# Phase 6 — Deployment and Synthesis

Deployment does **not** mean “put everything behind an API.”

Different analytical outputs have different deployment forms.

| Technique | Deployment artifact |
|---|---|
| Clustering | monthly customer-segment table/dashboard |
| Anomaly detection | ranked investigation queue |
| Supervised ML | risk-scoring pipeline |
| Association rules | merchandising/recommendation rule table |
| LSH | similarity retrieval index |

## Monitoring

### Clustering
- population share by cluster
- centroid/profile drift
- business outcome by segment

### Anomaly detection
- alert volume
- investigator confirmation rate
- recurring anomaly types

### Supervised model
- F1 / recall / precision
- calibration
- feature drift
- target drift

### Association rules
- rule support over time
- conversion after using rule
- seasonality

### LSH
- candidate count
- retrieval latency
- recall against exact neighbors on a sampled benchmark


# Clean Conclusion / Synthesis

This capstone intentionally uses several algorithms, but the final insight is **not** that e-commerce needs many models.

The useful synthesis is that each method answers a different layer of the same business system:

### 1. Clustering — **Who are our customers?**
RFM segmentation converts raw order history into behavioral groups that can support targeting and retention.

### 2. Anomaly detection — **What deserves attention?**
Isolation Forest creates a triage mechanism for unusually large, costly, or delayed orders.

### 3. Supervised learning — **What may happen next?**
Low-review prediction turns historical outcomes into an intervention signal—provided every feature is available at the intended scoring moment.

### 4. Association rule mining — **What tends to occur together?**
Support, confidence, and lift expose cross-sell opportunities that can be tested in merchandising experiments.

### 5. LSH — **Who or what is similar without exhaustive search?**
MinHash LSH turns set similarity into scalable approximate retrieval.

## The CRISP-DM lesson

The strongest part of this project is not any individual algorithm. It is the discipline of:

- defining the real decision first;
- respecting data grain;
- preventing leakage;
- creating task-specific representations;
- choosing metrics that match the method;
- testing whether results are operationally useful;
- documenting assumptions;
- monitoring deployed outputs;
- iterating back to earlier CRISP-DM phases when evidence changes the problem.

That is what turns a collection of techniques into an **end-to-end data science project**.



> **Quiz**
>
> 1. Which method in this capstone is best described as a retrieval method rather than a predictive model?
> 2. Which CRISP-DM phase should be revisited if stakeholders say the prediction arrives too late to act?
> 3. What is the single most important synthesis principle from the project?
>
> <details>
> <summary><b>Answer key</b></summary>
>
> 1. MinHash LSH.
> 2. Business Understanding, then Data Preparation/Modeling as needed.
> 3. Choose and connect methods based on the business decision; algorithms are tools inside the CRISP-DM process, not the objective.
>
> </details>


# Final Self-Assessment

Before calling the project complete, you should be able to explain:

- why CRISP-DM is iterative;
- the grain of every table before a join;
- the difference between clustering, anomaly detection, classification, association mining, and retrieval;
- why leakage is defined by the prediction moment;
- why K-Means needs scaling;
- why anomaly ≠ fraud;
- why imbalanced classification needs more than accuracy;
- why lift requires support;
- why MinHash approximates Jaccard;
- why LSH is approximate and sub-linear;
- how each output would be deployed and monitored.

If you can explain those points without looking at the notebook, you have learned the core methodology rather than only copied code.
